# Web Scraping with `urlopen()` and BeautifulSoup
Notebook with all 6 exercises solved. Each exercise is implemented as reusable functions to build a small scraping toolkit.

> **Note:** always respect each site's `robots.txt` and terms of use, and avoid sending too many requests in a row (the scripts include pauses).

## Setup

In [ ]:
!pip install beautifulsoup4 lxml --quiet

In [ ]:
import re, json, time, random, datetime
from collections import Counter
from pathlib import Path
from urllib.request import urlopen, Request
from urllib.parse import urljoin
from urllib.error import HTTPError, URLError

import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 200)

# Many sites (Wikipedia, IMDb, CISA) reject requests that have no User-Agent
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def fetch(url, timeout=30):
    """Download a URL with urlopen() and return its text (str)."""
    req = Request(url, headers=HEADERS)
    with urlopen(req, timeout=timeout) as response:
        charset = response.headers.get_content_charset() or "utf-8"
        return response.read().decode(charset, errors="replace")

## 🌟 Exercise 1: Parsing HTML with BeautifulSoup
We save the given HTML to a local file and read it with `urlopen()` (using a `file://` URL), so the exercise works without an internet connection.

In [ ]:
sports_html = r'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sports World</title>
    <style>
        body { font-family: Arial, sans-serif; }
        header, nav, section, article, footer { margin: 20px; padding: 15px; }
        nav { background-color: #333; }
        nav a { color: white; padding: 14px 20px; text-decoration: none; display: inline-block; }
        nav a:hover { background-color: #ddd; color: black; }
        .video { text-align: center; margin: 20px 0; }
    </style>
</head>
<body>

    <header>
        <h1>Welcome to Sports World</h1>
        <p>Your one-stop destination for the latest sports news and videos.</p>
    </header>

    <nav>
        <a href="#football">Football</a>
        <a href="#basketball">Basketball</a>
        <a href="#tennis">Tennis</a>
    </nav>

    <section id="football">
        <h2>Football</h2>
        <article>
            <h3>Latest Football News</h3>
            <p>Read about the latest football matches and player news.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/football-video-id" frameborder="0" allowfullscreen>
                </iframe>
            </div>
        </article>
    </section>

    <section id="basketball">
        <h2>Basketball</h2>
        <article>
            <h3>NBA Highlights</h3>
            <p>Watch highlights from the latest NBA games.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/basketball-video-id" frameborder="0" allowfullscreen>
                </iframe>
            </div>
        </article>
    </section>

    <section id="tennis">
        <h2>Tennis</h2>
        <article>
            <h3>Grand Slam Updates</h3>
            <p>Get the latest updates from the world of Grand Slam tennis.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/tennis-video-id" frameborder="0" allowfullscreen></iframe>
            </div>
        </article>
    </section>

    <footer>
        <form action="mailto:contact@sportsworld.com" method="post" enctype="text/plain">
            <label for="name">Name:</label><br>
            <input type="text" id="name" name="name"><br>
            <label for="email">Email:</label><br>
            <input type="email" id="email" name="email"><br>
            <label for="message">Message:</label><br>
            <textarea id="message" name="message" rows="4" cols="50"></textarea><br><br>
            <input type="submit" value="Send">
        </form>
    </footer>

</body>
</html>'''

Path("sports.html").write_text(sports_html, encoding="utf-8")
print("File sports.html created.")

In [ ]:
# 1) Read the HTML content of the page with urlopen()
file_url = Path("sports.html").resolve().as_uri()      # file:///content/sports.html
with urlopen(file_url) as response:
    html_content = response.read().decode("utf-8")
print(file_url)
print(html_content[:200], "...")

In [ ]:
# 2) Create the BeautifulSoup object
soup = BeautifulSoup(html_content, "html.parser")

# 3) Title of the page (content of the <title> tag)
print("Title:", soup.title.string)

# 4) All paragraphs (<p> tags)
paragraphs = soup.find_all("p")
print(f"\nParagraphs found: {len(paragraphs)}")
for i, p in enumerate(paragraphs, 1):
    print(f"  {i}. {p.get_text(strip=True)}")

# 5) All links (<a href="">)
links = [a["href"] for a in soup.find_all("a", href=True)]
print(f"\nLinks found: {len(links)}")
for link in links:
    print("  -", link)

In [ ]:
# Extra: link text with each URL, and the embedded videos (iframes)
print("Links with their text:", [(a.get_text(strip=True), a["href"]) for a in soup.find_all("a", href=True)])
print("Videos:", [f["src"] for f in soup.find_all("iframe")])

## 🌟 Exercise 2: Download Wikipedia's `robots.txt`

In [ ]:
def get_robots_txt(site="https://en.wikipedia.org"):
    return fetch(site.rstrip("/") + "/robots.txt")

robots = get_robots_txt()
print(f"Lines: {len(robots.splitlines())} | Characters: {len(robots)}\n")
print(robots)

In [ ]:
# (Optional) save the content to a file
Path("wikipedia_robots.txt").write_text(robots, encoding="utf-8")

## 🌟 Exercise 3: Extract the Header Tags (`h1`–`h6`) from Wikipedia's Main Page

In [ ]:
def extract_headers(url):
    soup = BeautifulSoup(fetch(url), "html.parser")
    headers = soup.find_all(re.compile(r"^h[1-6]$"))          # h1, h2, ..., h6
    return [(h.name, h.get_text(" ", strip=True)) for h in headers]

wiki_headers = extract_headers("https://en.wikipedia.org/wiki/Main_Page")

for tag, text in wiki_headers:
    print(f"<{tag}> {text}")

In [ ]:
# Summary table
df_headers = pd.DataFrame(wiki_headers, columns=["tag", "text"])
display(df_headers)
print(df_headers["tag"].value_counts().sort_index())

## 🌟 Exercise 4: Checking for a Page Title
The function accepts a **URL** or a raw **HTML string**, and considers that there is no title if the `<title>` tag is missing or empty.

In [ ]:
def has_title(source):
    """Return True if the page has a <title> with text. `source` can be a URL or an HTML string."""
    html = fetch(source) if source.strip().lower().startswith(("http://", "https://")) else source
    soup = BeautifulSoup(html, "html.parser")
    title_tag = soup.find("title")
    return bool(title_tag and title_tag.get_text(strip=True))

# Tests with local HTML
tests = {
    "With title":   "<html><head><title>My page</title></head><body>Hello</body></html>",
    "Empty title":  "<html><head><title>   </title></head><body>Hello</body></html>",
    "No title":     "<html><head></head><body>Hello</body></html>",
    "Sports page":  sports_html,
}
for name, html in tests.items():
    print(f"{name:12s} -> {has_title(html)}")

In [ ]:
# Test with a real page
url = "https://en.wikipedia.org/wiki/Main_Page"
print(url, "->", has_title(url))
print("Title:", BeautifulSoup(fetch(url), "html.parser").title.string)

## 🌟 Exercise 5: Analyzing US-CERT Security Alerts
US-CERT is now part of **CISA**; the old alerts (`us-cert.gov/ncas/alerts`) live under *Cybersecurity Alerts & Advisories*:
`https://www.cisa.gov/news-events/cybersecurity-advisories`

**Strategy:**
1. Detect the page's "Alert" filter (if available) so we do not crawl other advisory types.
2. Walk through the pages (sorted by date, newest first) reading *date + type* pairs.
3. Count the entries of type **Alert** from the current year; stop as soon as a previous year appears.

In [ ]:
CISA_URL = "https://www.cisa.gov/news-events/cybersecurity-advisories"
DATE_RE = re.compile(r"^([A-Z][a-z]{2}) (\d{2}), (\d{4})$")          # e.g. "Aug 26, 2025"
KNOWN_TYPES = {"Alert", "Analysis Report", "Cybersecurity Advisory", "ICS Advisory", "ICS Medical Advisory"}

def parse_advisories(html):
    """Return [(date, type), ...] for the entries listed on a CISA page."""
    soup = BeautifulSoup(html, "html.parser")
    lines = [l.strip() for l in soup.get_text("\n").splitlines() if l.strip()]
    entries = []
    for i, line in enumerate(lines[:-1]):
        m = DATE_RE.match(line)
        if m:
            kind = lines[i + 1].split("|")[0].strip()          # "Alert | TA10-021A" -> "Alert"
            if kind in KNOWN_TYPES:
                date = datetime.datetime.strptime(line, "%b %d, %Y").date()
                entries.append((date, kind))
    return entries

def find_alert_filter_url(html):
    """Look for the 'Alert' filter link (advisory_type facet) in the page."""
    soup = BeautifulSoup(html, "html.parser")
    for a in soup.find_all("a", href=True):
        if "advisory_type" in a["href"] and a.get_text(strip=True).lower().startswith("alert"):
            return urljoin(CISA_URL, a["href"])
    return None

def count_cert_alerts(year=None, max_pages=150, delay=1.0):
    year = year or datetime.date.today().year
    first_html = fetch(CISA_URL)
    base = find_alert_filter_url(first_html) or CISA_URL
    print("Base URL:", base)
    sep = "&" if "?" in base else "?"

    count, total_seen = 0, 0
    for page in range(max_pages):
        html = first_html if (page == 0 and base == CISA_URL) else fetch(f"{base}{sep}page={page}")
        entries = parse_advisories(html)
        if not entries:
            print(f"No entries on page {page}; stopping.")
            break
        total_seen += len(entries)
        count += sum(1 for d, k in entries if d.year == year and k == "Alert")
        if min(d for d, _ in entries).year < year:             # we have reached the previous year
            break
        time.sleep(delay)                                      # be polite to the server
    print(f"Entries reviewed: {total_seen} | pages: {page + 1}")
    return count

In [ ]:
# Offline test of the parsing logic with sample text (same format as the real CISA page)
sample_html = """
<div>Aug 26, 2025</div><div>Alert</div><h3>CISA Adds One Known Exploited Vulnerability to Catalog</h3>
<div>Aug 26, 2025</div><div>ICS Advisory | ICSA-25-238-01</div><h3>Some ICS product</h3>
<div>Aug 22, 2025</div><div>Alert</div><h3>CISA Requests Public Comment</h3>
<div>Dec 30, 2024</div><div>Alert</div><h3>Old alert</h3>
"""
print(parse_advisories(sample_html))

In [ ]:
current_year = datetime.date.today().year
n_alerts = count_cert_alerts(current_year)
print(f"\nSecurity alerts (type 'Alert') published in {current_year}: {n_alerts}")

> If the site changes its layout or blocks automated requests, `parse_advisories` is the only function that would need adjusting. To count *all* advisories of the year (not only `Alert`), simply remove the `k == "Alert"` condition.

## 🌟 Exercise 6: Scraping Movie Details
**Source:** IMDb's *Top 250* chart (`https://www.imdb.com/chart/top/`).

**Strategy:**
1. Read the Top 250 list (IMDb embeds the data in a JSON-LD `application/ld+json` block).
2. Pick 10 random movies.
3. Visit each movie's page and extract its **name, year and summary** (also from its JSON-LD).

In [ ]:
IMDB_TOP_URL = "https://www.imdb.com/chart/top/"

def load_json_ld(html):
    """Return all JSON-LD blocks of a page as a list of dicts."""
    soup = BeautifulSoup(html, "html.parser")
    blocks = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
            blocks.extend(data if isinstance(data, list) else [data])
        except json.JSONDecodeError:
            continue
    return blocks

def get_top_movies(html):
    """List of {'name', 'url'} from the Top 250 (JSON-LD first, /title/tt... links as fallback)."""
    movies = []
    for block in load_json_ld(html):
        for el in block.get("itemListElement", []):
            item = el.get("item", el)
            if item.get("url") and item.get("name"):
                movies.append({"name": item["name"], "url": item["url"]})
    if not movies:   # fallback: links found in the HTML
        soup = BeautifulSoup(html, "html.parser")
        seen = set()
        for a in soup.find_all("a", href=re.compile(r"/title/tt\d+")):
            url = urljoin("https://www.imdb.com", a["href"].split("?")[0])
            name = a.get_text(" ", strip=True)
            if name and url not in seen:
                seen.add(url)
                movies.append({"name": name, "url": url})
    return movies

def get_movie_details(html, fallback_name=""):
    """Extract name, year and summary from a movie page."""
    name, year, summary = fallback_name, None, None
    for block in load_json_ld(html):
        if block.get("@type") == "Movie":
            name = block.get("name") or name
            summary = block.get("description")
            if block.get("datePublished"):
                year = int(block["datePublished"][:4])
            break
    soup = BeautifulSoup(html, "html.parser")
    if year is None and soup.title:                                  # e.g. "Inception (2010) - IMDb"
        m = re.search(r"\((\d{4})\)", soup.title.get_text())
        year = int(m.group(1)) if m else None
    if not summary:
        meta = soup.find("meta", attrs={"name": "description"})
        summary = meta["content"] if meta and meta.get("content") else None
    return {"name": name, "year": year, "summary": summary}

In [ ]:
# Offline test of the logic with sample HTML
mock_list = """<script type="application/ld+json">{"@type":"ItemList","itemListElement":[
 {"@type":"ListItem","item":{"@type":"Movie","url":"https://www.imdb.com/title/tt0111161/","name":"The Shawshank Redemption"}},
 {"@type":"ListItem","item":{"@type":"Movie","url":"https://www.imdb.com/title/tt0068646/","name":"The Godfather"}}]}</script>"""
mock_movie = """<html><head><title>The Godfather (1972) - IMDb</title>
<script type="application/ld+json">{"@type":"Movie","name":"The Godfather","datePublished":"1972-03-24","description":"The aging patriarch of an organized crime dynasty transfers control to his reluctant son."}</script></head></html>"""
print(get_top_movies(mock_list))
print(get_movie_details(mock_movie))

In [ ]:
def scrape_random_movies(n=10, seed=None, delay=1.5):
    top_html = fetch(IMDB_TOP_URL)
    movies = get_top_movies(top_html)
    if not movies:
        raise RuntimeError("Could not read the Top 250 (IMDb may have blocked the request or changed its structure).")
    print(f"Movies found in the chart: {len(movies)}")

    rng = random.Random(seed)
    chosen = rng.sample(movies, min(n, len(movies)))
    rows = []
    for m in chosen:
        try:
            details = get_movie_details(fetch(m["url"]), fallback_name=m["name"])
        except (HTTPError, URLError) as e:
            details = {"name": m["name"], "year": None, "summary": f"Download error: {e}"}
        details["url"] = m["url"]
        rows.append(details)
        time.sleep(delay)                                            # be polite to the server
    return pd.DataFrame(rows)

movies_df = scrape_random_movies(n=10)
movies_df.index = movies_df.index + 1
display(movies_df[["name", "year", "summary"]])

In [ ]:
# Report built from the scraped data
valid_years = movies_df["year"].dropna().astype(int)
print("=== Summary of the 10 movies ===")
for i, row in movies_df.iterrows():
    print(f"{i:>2}. {row['name']} ({row['year']})")
print(f"\nOldest year   : {valid_years.min()}")
print(f"Newest year   : {valid_years.max()}")
print(f"Average year  : {valid_years.mean():.0f}")
print(f"Movies before 2000: {(valid_years < 2000).sum()}")

## Scraping Toolkit Summary
| Technique | Where it was used |
|---|---|
| `urlopen()` + `Request` with `User-Agent` | All exercises (`fetch`) |
| `BeautifulSoup(html, "html.parser")` | All |
| `soup.title`, `soup.find_all("p")`, `find_all("a", href=True)` | Ex. 1 and 4 |
| Regex tag search (`re.compile("^h[1-6]$")`) | Ex. 3 |
| Text and date parsing + pagination | Ex. 5 |
| Reading **JSON-LD** (`application/ld+json`) | Ex. 6 |
| Pauses (`time.sleep`) and error handling | Ex. 5 and 6 |